# Lecture recommendation — model pipeline (aligned with dropout prediction notebook)

Predict **recommended_subject** (weakest lecture track) from five numeric marks: Math, Physics, Chemistry, English, Computer_Science. Same layout as `student-dropout-prediction-pytorch-97-69.ipynb`: EDA → preprocessing → classical models → **PyTorch** feedforward → export **Random Forest** `joblib` for Java.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
DATA_CSV = NOTEBOOK_DIR / "student_course_marks.csv"

for root, _, files in os.walk(NOTEBOOK_DIR):
    for name in files:
        if name.endswith(".csv"):
            print(os.path.join(root, name))

In [ ]:
if not DATA_CSV.is_file():
    raise FileNotFoundError(
        f"Missing {DATA_CSV.name}. From this folder run: python generate_dummy_student_course_data.py"
    )

data = pd.read_csv(DATA_CSV)
data.head()

In [ ]:
data.info()

In [ ]:
data.isna().sum()

In [ ]:
SUBJECTS = ["Math", "Physics", "Chemistry", "English", "Computer_Science"]
TARGET = "recommended_subject"

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

X_df = data[SUBJECTS].copy()
y_raw = data[TARGET].astype(str)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)

le_y = LabelEncoder()
y_enc = le_y.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc,
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier,
    BaggingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "Bagging": BaggingClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=2000),
    "SVC": SVC(random_state=42),
    "XGBoost": XGBClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(random_state=42, verbose=False),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(name)
    print(classification_report(y_test, y_pred, target_names=le_y.classes_))
    print(confusion_matrix(y_test, y_pred))
    print("------------------------------------")

## PyTorch

Feedforward classifier mirroring dropout layout: **Input → 64 (ReLU) → 32 (ReLU) → num_classes** (no sigmoid; **CrossEntropyLoss** for multi-class).

In [ ]:
import torch
from torch import nn, optim

X_torch = torch.tensor(X_scaled, dtype=torch.float32)
y_torch = torch.tensor(y_enc, dtype=torch.long)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_torch, y_torch, test_size=0.2, random_state=42, stratify=y_torch.numpy()
)


class LectureRecommendationNet(nn.Module):
    def __init__(self, input_size: int, num_classes: int):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, num_classes)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)


num_classes = len(le_y.classes_)
torch_model = LectureRecommendationNet(X_tr.shape[1], num_classes)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(torch_model.parameters(), lr=0.05)
l2_lambda = 0.01

for epoch in range(5000):
    torch_model.train()
    optimizer.zero_grad()
    logits = torch_model(X_tr)
    loss = criterion(logits, y_tr) + l2_lambda * sum(
        p.pow(2.0).sum() for p in torch_model.parameters()
    )
    loss.backward()
    optimizer.step()

torch_model.eval()
with torch.no_grad():
    logits = torch_model(X_te)
    predicted = torch.argmax(logits, dim=1)
    accuracy = (predicted == y_te).float().mean()
    print(f"PyTorch test accuracy: {accuracy.item() * 100:.4f}%")

In [ ]:
import json

import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JOB = ARTIFACT_DIR / "lecture_recommendation_rf.joblib"

# Same split as scaled path; train RF on **raw** marks so predict_recommendation.py / Java match.
X_train_raw, X_test_raw, y_train_rf, y_test_rf = train_test_split(
    X_df.values.astype(float),
    y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc,
)

clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced_subsample",
    n_jobs=-1,
)
clf.fit(X_train_raw, y_train_rf)
y_pred_rf = clf.predict(X_test_raw)
acc = accuracy_score(y_test_rf, y_pred_rf)
print(f"Production RF test accuracy (raw marks): {acc:.4f}")
print(classification_report(y_test_rf, y_pred_rf, target_names=le_y.classes_))

bundle = {
    "model": clf,
    "feature_columns": SUBJECTS,
    "label_encoder": le_y,
    "metrics": {"test_accuracy": float(acc)},
}
joblib.dump(bundle, OUT_JOB)
(ARTIFACT_DIR / "lecture_recommendation_rf.meta.json").write_text(
    json.dumps(
        {
            "feature_columns": SUBJECTS,
            "classes": list(le_y.classes_),
            "test_accuracy": float(acc),
            "artifact": str(OUT_JOB),
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(f"Saved bundle for Java/predict_recommendation.py: {OUT_JOB}")